In [ ]:
# =========================================================
# Notebook 1 — Classificação + Probabilidade de Campeão
# (Jogos World Cup + FIFA ranking + flag "já foi campeão")
# =========================================================

# --------------------------
# 1) Imports
# --------------------------
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, log_loss

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

import joblib

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
# --------------------------
# 2) Helpers
# --------------------------
def norm_team_name(x: str) -> str:
    if pd.isna(x):
        return x
    return str(x).strip()

def find_col(df: pd.DataFrame, candidates):
    #Devolve a primeira coluna existente (case-insensitive) dentro de candidates.
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None

def safe_to_datetime(s):
    return pd.to_datetime(s, errors="coerce")


In [ ]:
# --------------------------
# 3) Ler datasets
# --------------------------
FIFA_PATH = "fifa_ranking.csv"
MATCHES_PATH = "worldcupmatches.csv"
CUPS_PATH = "worldcups.csv"

fifa = pd.read_csv(FIFA_PATH)
matches = pd.read_csv(MATCHES_PATH)
cups = pd.read_csv(CUPS_PATH)

print("FIFA:", fifa.shape)
print("Matches:", matches.shape)
print("Cups:", cups.shape)

display(fifa.head())
display(matches.head())
display(cups.head())


In [ ]:
# --------------------------
# 4) Detectar colunas e normalizar
# --------------------------
# FIFA
fifa_country_col = find_col(fifa, ["country_full", "country", "team", "team name", "name", "pais"])
fifa_rank_col = find_col(fifa, ["rank", "ranking", "position"])
fifa_points_col = find_col(fifa, ["total_points", "points"])
fifa_date_col = find_col(fifa, ["rank_date", "ranking_date", "date"])  # se existir

if fifa_country_col is None or fifa_rank_col is None:
    raise ValueError("No fifa_ranking.csv não encontrei colunas de país e rank.")

fifa[fifa_country_col] = fifa[fifa_country_col].apply(norm_team_name)
fifa[fifa_rank_col] = pd.to_numeric(fifa[fifa_rank_col], errors="coerce")

if fifa_points_col:
    fifa[fifa_points_col] = pd.to_numeric(fifa[fifa_points_col], errors="coerce")

if fifa_date_col:
    fifa[fifa_date_col] = safe_to_datetime(fifa[fifa_date_col])

# Matches (colunas que disseste que tens)
home_col = find_col(matches, ["Home Team Name"])
away_col = find_col(matches, ["Away Team Name"])
hgoals_col = find_col(matches, ["Home Team Goals"])
agoals_col = find_col(matches, ["Away Team Goals"])
stage_col = find_col(matches, ["Stage"])
year_col = find_col(matches, ["Year", "year", "Edition"])
date_col = find_col(matches, ["Datetime", "Date", "date", "match_date"])

for c in [home_col, away_col, hgoals_col, agoals_col, stage_col]:
    if c is None:
        raise ValueError("No worldcupmatches.csv faltam colunas esperadas (Stage/Home/Away/Goals).")

matches[home_col] = matches[home_col].apply(norm_team_name)
matches[away_col] = matches[away_col].apply(norm_team_name)

if date_col:
    matches[date_col] = safe_to_datetime(matches[date_col])
if year_col:
    matches[year_col] = pd.to_numeric(matches[year_col], errors="coerce").astype("Int64")

# Cups
winner_col = find_col(cups, ["Winner", "winner", "Champion", "champion"])
cup_year_col = find_col(cups, ["Year", "year", "Edition"])

if winner_col is None:
    raise ValueError("No worldcups.csv não encontrei a coluna Winner/Champion. Confirma o nome.")

cups[winner_col] = cups[winner_col].apply(norm_team_name)
if cup_year_col:
    cups[cup_year_col] = pd.to_numeric(cups[cup_year_col], errors="coerce").astype("Int64")

print("Colunas detetadas:")
print("FIFA:", fifa_country_col, fifa_rank_col, fifa_points_col, fifa_date_col)
print("Matches:", home_col, away_col, hgoals_col, agoals_col, stage_col, year_col, date_col)
print("Cups:", winner_col, cup_year_col)


In [ ]:
# --------------------------
# 5) Criar target y (resultado do jogo) + base features
# --------------------------
# Limpar linhas sem golos
matches = matches.dropna(subset=[hgoals_col, agoals_col, home_col, away_col, stage_col]).copy()

matches[hgoals_col] = pd.to_numeric(matches[hgoals_col], errors="coerce")
matches[agoals_col] = pd.to_numeric(matches[agoals_col], errors="coerce")
matches = matches.dropna(subset=[hgoals_col, agoals_col]).copy()

# Target: 0=AwayWin, 1=Draw, 2=HomeWin
goal_diff = matches[hgoals_col] - matches[agoals_col]
matches["y"] = np.where(goal_diff > 0, 2, np.where(goal_diff < 0, 0, 1))

matches["stage"] = matches[stage_col].astype(str).str.strip()

# Equipas (para merge e para flags)
matches["home_team"] = matches[home_col]
matches["away_team"] = matches[away_col]

print("Distribuição do target (0=Away,1=Draw,2=Home):")
print(pd.Series(matches["y"]).value_counts(normalize=True).sort_index())


In [ ]:
# --------------------------
# 6) Feature: "já foi campeão?"
# (Se houver ano nos jogos e no cups, usamos "antes do ano do jogo")
# --------------------------
champions_ever = set(cups[winner_col].dropna().unique().tolist())

if cup_year_col is not None:
    champion_year_map = cups[[cup_year_col, winner_col]].dropna().copy()
else:
    champion_year_map = None

def champion_before_year(team: str, year: int) -> int:
    if champion_year_map is None or pd.isna(year):
        return int(team in champions_ever)
    wins = champion_year_map[(champion_year_map[winner_col] == team) & (champion_year_map[cup_year_col] < year)]
    return int(len(wins) > 0)

matches["home_champion_ever"] = matches["home_team"].apply(lambda t: int(t in champions_ever))
matches["away_champion_ever"] = matches["away_team"].apply(lambda t: int(t in champions_ever))

if year_col is not None and champion_year_map is not None:
    matches["home_champion_before"] = matches.apply(
        lambda r: champion_before_year(r["home_team"], int(r[year_col]) if not pd.isna(r[year_col]) else np.nan), axis=1
    )
    matches["away_champion_before"] = matches.apply(
        lambda r: champion_before_year(r["away_team"], int(r[year_col]) if not pd.isna(r[year_col]) else np.nan), axis=1
    )
else:
    matches["home_champion_before"] = matches["home_champion_ever"]
    matches["away_champion_before"] = matches["away_champion_ever"]

display(matches[["home_team","away_team","home_champion_before","away_champion_before","stage","y"]].head(10))


In [ ]:
# --------------------------
# 7) Preparar ranking FIFA (snapshot ou as-of se tiver data)
# --------------------------
# Tabela FIFA simplificada
cols = [fifa_country_col, fifa_rank_col]
if fifa_points_col:
    cols.append(fifa_points_col)
if fifa_date_col:
    cols.append(fifa_date_col)

fifa_small = fifa[cols].dropna(subset=[fifa_country_col, fifa_rank_col]).copy()
fifa_small = fifa_small.rename(columns={
    fifa_country_col: "team",
    fifa_rank_col: "fifa_rank"
})
if fifa_points_col:
    fifa_small = fifa_small.rename(columns={fifa_points_col: "fifa_points"})
if fifa_date_col:
    fifa_small = fifa_small.rename(columns={fifa_date_col: "rank_date"})
    fifa_small["rank_date"] = safe_to_datetime(fifa_small["rank_date"])

fifa_small["team"] = fifa_small["team"].apply(norm_team_name)
fifa_small["fifa_rank"] = pd.to_numeric(fifa_small["fifa_rank"], errors="coerce")
if "fifa_points" in fifa_small.columns:
    fifa_small["fifa_points"] = pd.to_numeric(fifa_small["fifa_points"], errors="coerce")

# Se tiver rank_date e também data/ano do jogo - merge as-of;
if "rank_date" in fifa_small.columns and (date_col is not None or year_col is not None):
    if date_col is not None:
        matches["match_date"] = matches[date_col]
    else:
        matches["match_date"] = pd.to_datetime(matches[year_col].astype(str) + "-07-01", errors="coerce")

    fifa_small = fifa_small.dropna(subset=["rank_date"]).sort_values(["team", "rank_date"])

    def merge_asof_team(df_matches, team_col_name, prefix):
        tmp = df_matches[[team_col_name, "match_date"]].copy()
        tmp = tmp.rename(columns={team_col_name: "team"}).sort_values(["team","match_date"])

        out = pd.merge_asof(
            tmp,
            fifa_small.sort_values(["team","rank_date"]),
            left_on="match_date",
            right_on="rank_date",
            by="team",
            direction="backward"
        )

        out = out.drop(columns=["team"])
        out = out.rename(columns={
            "fifa_rank": f"{prefix}_rank",
            "fifa_points": f"{prefix}_points" if "fifa_points" in out.columns else "fifa_points"
        })
        return out

    home_rank = merge_asof_team(matches, "home_team", "home")
    away_rank = merge_asof_team(matches, "away_team", "away")

    matches = pd.concat([matches.reset_index(drop=True),
                         home_rank.reset_index(drop=True),
                         away_rank.reset_index(drop=True)], axis=1)
else:
    fifa_snap = fifa_small.sort_values("fifa_rank").drop_duplicates("team", keep="first").copy()

    # Home
    home_cols = {"team": "home_team", "fifa_rank": "home_rank"}
    if "fifa_points" in fifa_snap.columns:
        home_cols["fifa_points"] = "home_points"
    matches = matches.merge(fifa_snap.rename(columns=home_cols), on="home_team", how="left")

    # Away
    away_cols = {"team": "away_team", "fifa_rank": "away_rank"}
    if "fifa_points" in fifa_snap.columns:
        away_cols["fifa_points"] = "away_points"
    matches = matches.merge(fifa_snap.rename(columns=away_cols), on="away_team", how="left")

# Diferenças (features fortes)
matches["rank_diff"] = matches["away_rank"] - matches["home_rank"]
if "home_points" in matches.columns and "away_points" in matches.columns:
    matches["points_diff"] = matches["home_points"] - matches["away_points"]
else:
    matches["points_diff"] = np.nan

# Remover jogos sem ranking
matches = matches.dropna(subset=["home_rank","away_rank"]).copy()

print("Após merge ranking:", matches.shape)
display(matches[["home_team","away_team","home_rank","away_rank","rank_diff","stage","y"]].head(10))


In [ ]:
# --------------------------
# 8) Dataset final (X, y)
# --------------------------
feature_cols_num = [
    "home_rank", "away_rank", "rank_diff",
    "home_champion_before", "away_champion_before"
]
if "points_diff" in matches.columns:
    feature_cols_num.append("points_diff")

feature_cols_cat = ["stage"]

X = matches[feature_cols_num + feature_cols_cat].copy()
y = matches["y"].astype(int).copy()

print("X shape:", X.shape, "| y distribution:", np.bincount(y))
display(X.head())


In [ ]:
# --------------------------
# 9) Split treino/teste (por ano se existir)
# --------------------------
if year_col is not None:
    groups = matches[year_col].fillna(-1).astype(int).values
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
    train_idx, test_idx = next(gss.split(X, y, groups=groups))
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
    )

print("Train:", X_train.shape, "| Test:", X_test.shape)


In [ ]:
# --------------------------
# 10) Pré-processamento (Pipeline)
# --------------------------
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, feature_cols_num),
        ("cat", categorical_transformer, feature_cols_cat),
    ]
)

def eval_model(name, pipe, Xtr, ytr, Xte, yte):
    pipe.fit(Xtr, ytr)
    pred = pipe.predict(Xte)

    acc = accuracy_score(yte, pred)
    f1m = f1_score(yte, pred, average="macro")

    print(f"\n=== {name} ===")
    print("Accuracy:", round(acc, 4), "| F1-macro:", round(f1m, 4))
    print("Confusion matrix:\n", confusion_matrix(yte, pred))
    print("\nClassification report:\n", classification_report(yte, pred, digits=4))

    if hasattr(pipe, "predict_proba"):
        proba = pipe.predict_proba(Xte)
        try:
            ll = log_loss(yte, proba, labels=[0,1,2])
            print("LogLoss:", round(ll, 4))
        except Exception:
            pass

    return pipe


In [ ]:
# --------------------------
# 11) Modelos (baseline + 3)
# --------------------------
baseline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", DummyClassifier(strategy="most_frequent"))
])

logreg = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", LogisticRegression(max_iter=2000))
])

rf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", RandomForestClassifier(
        n_estimators=400,
        random_state=RANDOM_SEED,
        class_weight="balanced_subsample"
    ))
])

gb = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", GradientBoostingClassifier(random_state=RANDOM_SEED))
])

_ = eval_model("Baseline (most frequent)", baseline, X_train, y_train, X_test, y_test)
_ = eval_model("Logistic Regression", logreg, X_train, y_train, X_test, y_test)
_ = eval_model("Random Forest", rf, X_train, y_train, X_test, y_test)
_ = eval_model("Gradient Boosting", gb, X_train, y_train, X_test, y_test)


In [ ]:
# --------------------------
# 12) Tuning (GridSearch) no RandomForest
# --------------------------
param_grid = {
    "clf__n_estimators": [300, 600],
    "clf__max_depth": [None, 10, 20],
    "clf__min_samples_split": [2, 5],
    "clf__min_samples_leaf": [1, 2],
}

grid = GridSearchCV(
    rf,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)
print("\nBest params:", grid.best_params_)
print("Best CV score (F1-macro):", round(grid.best_score_, 4))

best_model = grid.best_estimator_
_ = eval_model("Best Tuned RandomForest (test)", best_model, X_train, y_train, X_test, y_test)

joblib.dump(best_model, "best_worldcup_model.joblib")
print("\nModelo guardado: best_worldcup_model.joblib")


In [ ]:
# --------------------------
# 13) Preparar tabela de forças (Top 32 do ranking)
# --------------------------
# (Se o teu FIFA tiver data, isto vai usar o "melhor" ranking disponível;
#  aqui assumimos snapshot para simulação.)

def build_strength_table_from_fifa(fifa_df):
    ccol = find_col(fifa_df, ["team", fifa_country_col, "country_full", "country", "name", "pais"])
    rcol = find_col(fifa_df, ["fifa_rank", fifa_rank_col, "rank", "ranking", "position"])
    pcol = find_col(fifa_df, ["fifa_points", fifa_points_col, "total_points", "points"])

    tmp = fifa_df.copy()
    tmp = tmp.rename(columns={ccol: "team", rcol: "rank"})
    tmp["team"] = tmp["team"].apply(norm_team_name)
    tmp["rank"] = pd.to_numeric(tmp["rank"], errors="coerce")

    if pcol:
        tmp = tmp.rename(columns={pcol: "points"})
        tmp["points"] = pd.to_numeric(tmp["points"], errors="coerce")
    else:
        tmp["points"] = np.nan

    tmp = tmp.dropna(subset=["team", "rank"]).sort_values("rank").drop_duplicates("team", keep="first")
    return tmp[["team", "rank", "points"]].reset_index(drop=True)

strength = build_strength_table_from_fifa(fifa_small.rename(columns={
    "team": "team",
    "fifa_rank": "rank",
    **({"fifa_points": "points"} if "fifa_points" in fifa_small.columns else {})
}))

teams32 = strength.sort_values("rank").head(32)["team"].tolist()
print("Equipas usadas (Top 32 FIFA rank):")
display(pd.DataFrame({"team": teams32}))


In [ ]:
# --------------------------
# 14) Funções de simulação P(campeão)
# --------------------------
def build_match_features(teamA, teamB, stage_name="Group Stage"):
    teamA = norm_team_name(teamA)
    teamB = norm_team_name(teamB)

    ra = strength.loc[strength["team"] == teamA, "rank"]
    rb = strength.loc[strength["team"] == teamB, "rank"]
    if ra.empty or rb.empty:
        return None

    home_rank = float(ra.iloc[0])
    away_rank = float(rb.iloc[0])

    row = {
        "home_rank": home_rank,
        "away_rank": away_rank,
        "rank_diff": away_rank - home_rank,
        "home_champion_before": int(teamA in champions_ever),
        "away_champion_before": int(teamB in champions_ever),
        "stage": stage_name
    }

    # points_diff se existir
    if "points" in strength.columns and strength["points"].notna().any():
        pa = strength.loc[strength["team"] == teamA, "points"]
        pb = strength.loc[strength["team"] == teamB, "points"]
        if not pa.empty and not pb.empty:
            row["points_diff"] = float(pa.iloc[0]) - float(pb.iloc[0])
        else:
            row["points_diff"] = np.nan
    else:
        row["points_diff"] = np.nan

    return row

def simulate_single_match(model, teamA, teamB, stage_name):
    feat = build_match_features(teamA, teamB, stage_name=stage_name)
    if feat is None:
        # fallback se faltar ranking
        return np.random.choice([0,1,2], p=[0.35,0.30,0.35])

    Xrow = pd.DataFrame([feat])
    proba = model.predict_proba(Xrow)[0]  # labels [0,1,2]
    return np.random.choice([0,1,2], p=proba)

def simulate_group(model, teams, stage_name="Group Stage"):
    pts = {t: 0 for t in teams}

    for i in range(len(teams)):
        for j in range(i+1, len(teams)):
            A, B = teams[i], teams[j]
            outcome = simulate_single_match(model, A, B, stage_name)
            if outcome == 2:
                pts[A] += 3
            elif outcome == 0:
                pts[B] += 3
            else:
                pts[A] += 1
                pts[B] += 1

    items = list(pts.items())
    np.random.shuffle(items)  # desempate aleatório
    items.sort(key=lambda x: x[1], reverse=True)
    return [t for t,_ in items]

def simulate_knockout(model, teams, round_name):
    winners = []
    for i in range(0, len(teams), 2):
        A, B = teams[i], teams[i+1]
        outcome = simulate_single_match(model, A, B, round_name)

        if outcome == 2:
            winners.append(A)
        elif outcome == 0:
            winners.append(B)
        else:
            # empate -> penáltis: ligeira vantagem para melhor rank
            ra = float(strength.loc[strength["team"] == A, "rank"].iloc[0])
            rb = float(strength.loc[strength["team"] == B, "rank"].iloc[0])
            pA = 0.5 + (rb - ra) * 0.005
            pA = min(max(pA, 0.35), 0.65)
            winners.append(A if np.random.rand() < pA else B)

    return winners

def simulate_worldcup_32(model, teams32):
    teams32 = [norm_team_name(t) for t in teams32]
    teams32 = teams32.copy()
    np.random.shuffle(teams32)

    groups = [teams32[i*4:(i+1)*4] for i in range(8)]
    qualified = []

    # 8 grupos -> top2
    group_orders = []
    for g in groups:
        order = simulate_group(model, g, "Group Stage")
        group_orders.append(order)
        qualified.extend(order[:2])

    # Oitavos: A1 vs B2; B1 vs A2; ...
    r16 = []
    for k in range(0, 8, 2):
        A = group_orders[k]
        B = group_orders[k+1]
        r16 += [A[0], B[1], B[0], A[1]]

    qf = simulate_knockout(model, r16, "Round of 16")
    sf = simulate_knockout(model, qf, "Quarter-finals")
    fin = simulate_knockout(model, sf, "Semi-finals")
    champ = simulate_knockout(model, fin, "Final")[0]
    return champ

def champion_probabilities(model, teams, n_sims=5000):
    counts = {t: 0 for t in teams}
    for _ in range(n_sims):
        champ = simulate_worldcup_32(model, teams)
        counts[champ] += 1

    out = pd.DataFrame({"team": list(counts.keys()), "wins": list(counts.values())})
    out["p_champion"] = out["wins"] / n_sims
    out = out.sort_values("p_champion", ascending=False).reset_index(drop=True)
    return out


In [ ]:
# --------------------------
# 15) Correr simulação e ver Top 10
# --------------------------
# best_model = joblib.load("best_worldcup_model.joblib")

probs = champion_probabilities(best_model, teams32, n_sims=3000)  # aumenta para 10000 para mais estabilidade
display(probs.head(10))


In [ ]:
# --------------------------
# 16) Ver probabilidade de um país específico
# --------------------------
def prob_of_country(df_probs, country_name):
    country_name = norm_team_name(country_name)
    row = df_probs[df_probs["team"] == country_name]
    if row.empty:
        return None
    return float(row["p_champion"].iloc[0])

pais = teams32[0]
print(f"Probabilidade estimada de '{pais}' ser campeão:", prob_of_country(probs, pais))


In [ ]:
# --------------------------
# 17) Exportar resultados (opcional)
# --------------------------
probs.to_csv("champion_probabilities_top32.csv", index=False)
print("Guardado: champion_probabilities_top32.csv")
